# Data Exploration
**Where to run:** Kaggle Notebooks is easiest (attach the competition data, free GPU: Settings > Accelerator > GPU). Colab also works if you upload the data (see the Colab note in the data cell).

## 1. Import the dataset

In [1]:
# Install only if missing (timm gives easy access to pretrained models later)
import importlib, subprocess, sys
for pkg in ["timm", "scikit-learn"]:
    try:
        importlib.import_module(pkg.replace("scikit-learn", "sklearn"))
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

In [2]:
import random, glob
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split

SEED = 42
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| torch", torch.__version__)

Device: cpu | torch 2.14.0+cpu


## 2. Locate the data


In [9]:
DATA_DIR = Path(".")
TRAIN_DIR = DATA_DIR / "train" / "train"
TEST_DIR = DATA_DIR / "test" / "test"
print("DATA_DIR:", DATA_DIR.resolve())
print("train files:", len(list(TRAIN_DIR.iterdir())))
print("test files: ", len(list(TEST_DIR.iterdir())))
print([p.name for p in TRAIN_DIR.iterdir()][:3])

DATA_DIR: C:\Users\yzsha\Documents\DataAndAI_project\KaggleCompetitionProject\KaggleCompetitionENGE707
train files: 63660
test files:  29443
['00001.jpg', '00002.jpg', '00003.jpg']


## 3. Load the files

In [10]:
# IMPORTANT: read ids as strings so '00001' keeps its leading zeros
train_df = pd.read_csv(DATA_DIR / "train_labels.csv", dtype={"image_id": str})
sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv", dtype={"image_id": str})
print(train_df.shape, sample_sub.shape)
display(train_df.head())

# Use the exact class spellings from the data (brief says both CatEye and CatEyes)
CLASSES = sorted(train_df["class"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}
print(CLASS_TO_IDX)
print("Sample submission classes:", sorted(sample_sub["class"].unique()))

(63660, 2) (29443, 2)


,image_id,class
0,00001,DogEye
1,00002,DogMouth
2,00003,DogFace
3,00004,DogNose
4,00005,DogNose


{'CatEye': 0, 'CatFace': 1, 'CatMouth': 2, 'CatNose': 3, 'DogEye': 4, 'DogFace': 5, 'DogMouth': 6, 'DogNose': 7}
Sample submission classes: ['CatEye', 'CatFace', 'CatMouth', 'CatNose', 'DogEye', 'DogFace', 'DogMouth', 'DogNose']


## 4. Map the csv file to the image

In [13]:
# Map image_id -> file path, matching on the integer value of the id
# (sample_submission ids are unpadded, e.g. '1', but filenames are padded, e.g. '00001.jpg')
def build_path_map(folder):
    return {int(p.stem): p for p in folder.iterdir() if p.is_file()}

train_map, test_map = build_path_map(TRAIN_DIR), build_path_map(TEST_DIR)
train_df["path"] = train_df["image_id"].astype(int).map(train_map)
sample_sub["path"] = sample_sub["image_id"].astype(int).map(test_map)
print("Missing train files:", train_df["path"].isna().sum())
print("Missing test files: ", sample_sub["path"].isna().sum())
train_df["label"] = train_df["class"].map(CLASS_TO_IDX)

Missing train files: 0
Missing test files:  0


## 5. Structure of Data

In [15]:
# --- Structure of the training data ---
print("Train rows, columns:", train_df[["image_id", "class"]].shape)
print("Column dtypes:", train_df[["image_id", "class"]].dtypes.astype(str).to_dict())
print("Null values:", train_df[["image_id", "class"]].isna().sum().to_dict())
print("Duplicate ids in CSV:", train_df["image_id"].duplicated().sum())
print("Train id range:", train_df["image_id"].astype(int).min(), "-", train_df["image_id"].astype(int).max())
print("Train file types:", pd.Series([p.suffix.lower() for p in train_map.values()]).value_counts().to_dict())

# --- Structure of the test data ---
print("\nTest rows, columns:", sample_sub[["image_id", "class"]].shape)
print("Duplicate ids in sample_submission:", sample_sub["image_id"].duplicated().sum())
print("Test id range:", sample_sub["image_id"].astype(int).min(), "-", sample_sub["image_id"].astype(int).max())
print("Test file types:", pd.Series([p.suffix.lower() for p in test_map.values()]).value_counts().to_dict())
print("Test classes in sample_submission:", sample_sub["class"].value_counts().to_dict())

# Are the two sets kept separate? (same numbers are reused, but in different folders)
print("\nTrain files:", len(train_map), "| Test files:", len(test_map))
display(sample_sub[["image_id", "class"]].head())

Train rows, columns: (63660, 2)
Column dtypes: {'image_id': 'str', 'class': 'str'}
Null values: {'image_id': 0, 'class': 0}
Duplicate ids in CSV: 0
Train id range: 1 - 63660
Train file types: {'.jpg': 63660}

Test rows, columns: (29443, 2)
Duplicate ids in sample_submission: 0
Test id range: 1 - 29443
Test file types: {'.jpg': 29443}
Test classes in sample_submission: {'CatEye': 4567, 'DogEye': 4221, 'CatFace': 3571, 'CatMouth': 3570, 'CatNose': 3457, 'DogFace': 3407, 'DogMouth': 3341, 'DogNose': 3309}

Train files: 63660 | Test files: 29443


,image_id,class
0,1,DogEye
1,2,CatNose
2,3,CatNose
3,4,DogFace
4,5,DogMouth
